# Writing MongoDB queries

As you have seen, the MongoDB Query API is the mechanism with which you (or AI agents) interact with data stored in MongoDB.

While an LLM is responsible for generating these queries from natural language in text-to-query agents, it is important to understand what is happening under the hood.

In this exercise, you will perform some queries against the sample movies dataset using the MongoDB Query API.

**Run the cells below to install `pymongo` and create your MongoDB client.**

In [14]:
!pip install  --quiet pymongo==4.13.2


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [15]:
import os
from pymongo import MongoClient

MONGODB_URI = os.environ["MONGODB_URI"]
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI)

The movies dataset is actually a database consisting of multiple collections that you can write queries against.

Let's first analyze what collections are available in this database.

In `pymongo`, MongoDB's Python driver, you can access a database using dictionary keys.

**Complete the code below to access the `sample_mflix` database using dictionary-style access on the `mongodb_client`.**

In [16]:
# Access the sample_mflix database
db = mongodb_client["sample_mflix"]

You can list the names of collections in a MongoDB database using the `list_collection_names()` method of the database.

**Complete the code below to list the names of collection in the `sample_mflix` database.**

In [17]:
db.list_collection_names()

['movies', 'comments', 'sessions', 'theaters', 'users', 'embedded_movies']

Looking at the output of the above command, the `sample_mflix` database has the following collections:

* `comments`: Contains comments associated with specific movies.
* `movies`: Contains movie information, including release year, director, and reviews.
* `embedded_movies`: Contains details about a subset of movies in the `movies` collection, with additional embedding fields added to the documents to facilitate vector search.
* `users`: Contains user information.
* `theaters`: Contains locations of movie theaters.
* `sessions`: Contains comments associated with specific movies.

For now, let's pick the `movies` collection and perform some queries against it.

**Complete the code below to access the `movies` collection in the `sample_mflix` database.**

In [18]:
collection = db["movies"]

### Basic CRUD operations

The simplest way to interact with your MongoDB data is via CRUD (Create, Read, Update, and Delete) operations.

Let's try out some of the CRUD operations you came across in the previous video, using MongoDB's Python driver (`pymongo`).

To preview a random sample document in the `movies` collection, we will call the `find_one()` method from the Python driver with an empty query filter (`{}`).

![](images/find_one.png)

**Use the `find_one()` method of the `collection` to preview a sample document.**

In [19]:
collection.find_one({})

{'_id': ObjectId('573a1390f29313caabcd42e8'),
 'plot': 'A group of bandits stage a brazen train hold-up, only to find a determined posse hot on their heels.',
 'genres': ['Short', 'Western'],
 'runtime': 11,
 'cast': ['A.C. Abadie',
  "Gilbert M. 'Broncho Billy' Anderson",
  'George Barnes',
  'Justus D. Barnes'],
 'poster': 'https://m.media-amazon.com/images/M/MV5BMTU3NjE5NzYtYTYyNS00MDVmLWIwYjgtMmYwYWIxZDYyNzU2XkEyXkFqcGdeQXVyNzQzNzQxNzI@._V1_SY1000_SX677_AL_.jpg',
 'title': 'The Great Train Robbery',
 'fullplot': "Among the earliest existing films in American cinema - notable as the first film that presented a narrative story to tell - it depicts a group of cowboy outlaws who hold up a train and rob the passengers. They are then pursued by a Sheriff's posse. Several scenes have color included - all hand tinted.",
 'languages': ['English'],
 'released': datetime.datetime(1903, 12, 1, 0, 0),
 'directors': ['Edwin S. Porter'],
 'rated': 'TV-G',
 'awards': {'wins': 1, 'nominations': 0, 

Let's try a query with a simple query filter.

To find all movies that were released before the year 1905, we will call the `find()` method with a query filter.

The filter is essentially a Python dictionary where the key is the field in the MongoDB documents that you want to filter on (`year`), and the value is the filter expression, which uses the `$lt` operator to convey `< 1905`.

![](images/less_than.png)

Note that the `find()` method returns a cursor that you need to iterate over, to access the returned documents.

**Create the query filter using the `$lt` operator.**

In [20]:
old_movies_query = {"year":{"$lt":1905}}

In [21]:
import pprint

# Execute the query and iterate through the resulting cursor
for doc in collection.find(old_movies_query):
    pprint.pprint(doc)

{'_id': ObjectId('573a1390f29313caabcd42e8'),
 'awards': {'nominations': 0, 'text': '1 win.', 'wins': 1},
 'cast': ['A.C. Abadie',
          "Gilbert M. 'Broncho Billy' Anderson",
          'George Barnes',
          'Justus D. Barnes'],
 'countries': ['USA'],
 'directors': ['Edwin S. Porter'],
 'fullplot': 'Among the earliest existing films in American cinema - notable '
             'as the first film that presented a narrative story to tell - it '
             'depicts a group of cowboy outlaws who hold up a train and rob '
             "the passengers. They are then pursued by a Sheriff's posse. "
             'Several scenes have color included - all hand tinted.',
 'genres': ['Short', 'Western'],
 'imdb': {'id': 439, 'rating': 7.4, 'votes': 9847},
 'languages': ['English'],
 'lastupdated': '2015-08-13 00:27:59.177000000',
 'num_mflix_comments': 0,
 'plot': 'A group of bandits stage a brazen train hold-up, only to find a '
         'determined posse hot on their heels.',
 'poster'

Now, let's try a query with multiple conditions in the query filter. For this, we will use the `$and` operator, similar to the `AND` operator in SQL.

The `$and` operator accepts a list of query expressions, so in this case we have one expression to convey `year` < 1930 using the `$lt` operator, and another for `imdb.rating` >= 8 using the `$gte` operator.

![](images/multiple_conditions.png)

**Create a query to find movies that were released before 1930 and have an IMDB rating of at least 8 using the `$and`, `$lt` and `$gte` operators.**

In [22]:
best_old_movies_query = { "$and":[
                    {"year": {"$lt":1930}},
                    {"imdb.rating": {"$gte":8}}
                  ]
        }

In [23]:
# Execute the query and iterate through the resulting cursor
for doc in collection.find(best_old_movies_query):
    pprint.pprint(doc)

{'_id': ObjectId('573a1391f29313caabcd6e2a'),
 'awards': {'nominations': 0, 'text': '1 win.', 'wins': 1},
 'cast': ['Buster Keaton', 'Sybil Seely'],
 'countries': ['USA'],
 'directors': ['Edward F. Cline', 'Buster Keaton'],
 'fullplot': 'Buster and Sybil exit a chapel as newlyweds. Among the gifts is '
             "a portable house you easily put together in one week. It doesn't "
             "help that Buster's rival for Sybil switches the numbers on the "
             'crates containing the house parts.',
 'genres': ['Short', 'Comedy'],
 'imdb': {'id': 11541, 'rating': 8.3, 'votes': 3942},
 'languages': ['English'],
 'lastupdated': '2015-05-07 01:07:01.633000000',
 'num_mflix_comments': 0,
 'plot': 'A newly wedded couple attempt to build a house with a prefabricated '
         "kit, unaware that a rival sabotaged the kit's component numbering.",
 'rated': 'TV-G',
 'released': datetime.datetime(1920, 9, 1, 0, 0),
 'runtime': 25,
 'title': 'One Week',
 'tomatoes': {'lastUpdated': dat